# EM619 - Machine Learning
## Assignment 3: Neural Networks - MLPs in PyTorch
**Total Marks: 20** | **Questions: 5 (4 marks each)**

### Instructions
- Fill in every `# TODO` block. Do **not** delete function signatures.
- Every question has a mandatory **Observation** markdown cell which you need to answer in 2 to 4 crisp bullet points.
- Leave every `torch.manual_seed(...)` call exactly where it is - they are what make your numbers and plots reproducible for grading.
- Everything here runs on a **CPU** in a few minutes. No GPU needed.
- Run **Runtime -> Restart session and run all** before submitting; make sure your notebook is free from errors before submission.
- Rename this file to `<name>_EM619_A3.ipynb` before submitting.

| Q | Topic | Marks |
|:-:|:------|:-:|
| 1 | Why a network needs a nonlinearity | 4 |
| 2 | Inside the hidden layer - what one neuron computes | 4 |
| 3 | How many neurons, how many layers | 4 |
| 4 | Activation functions and where gradients die | 4 |
| 5 | Mini-batches - an epoch is not an update | 4 |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (6, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 42
torch.manual_seed(SEED)
bce = nn.BCEWithLogitsLoss()
mse = nn.MSELoss()

print("PyTorch version:", torch.__version__)

### Helpers: datasets, plotting and a training loop

The next cell is **given**. It contains the toy datasets, the plotting helpers and a full-batch
training loop that every question reuses. You do not need to read the plotting code - just run the cell.

- `make_xor`, `make_circles`, `make_moons`, `make_spiral` return `X` of shape `(N, 2)` and `y` of shape `(N,)` with 0/1 labels.
- `wave(x)` is the 1-D function used for the regression question.
- `plot_probability_surface(prob_fn, X, y, ax)` colours the plane by $P(y=1\mid x)$ and draws the 0.5 contour in black.
- `train(model, X, y, loss_fn, epochs, lr)` runs **full-batch Adam** and returns the loss of every epoch.
- `accuracy(model, X, y)` and `n_params(model)` do what their names say.


In [ ]:
# ============================ GIVEN CODE - just run it ============================
from math import pi

# ---------- datasets ----------
def make_xor(n=400, noise=0.25, seed=0):
    """Four corners at (+-1, +-1); class 1 in quadrants I and III."""
    torch.manual_seed(seed)
    X = torch.randint(0, 2, (n, 2)).float() * 2 - 1
    y = (X[:, 0] * X[:, 1] > 0).float()
    return X + noise * torch.randn(n, 2), y


def make_circles(n=400, noise=0.08, factor=0.45, seed=0):
    """Outer ring (class 0) around an inner disc (class 1)."""
    torch.manual_seed(seed)
    half = n // 2
    angle = 2 * pi * torch.rand(n)
    radius = torch.cat([torch.ones(half), factor * torch.ones(n - half)])
    X = torch.stack([radius * torch.cos(angle), radius * torch.sin(angle)], dim=1)
    y = torch.cat([torch.zeros(half), torch.ones(n - half)])
    return X + noise * torch.randn(n, 2), y


def make_moons(n=400, noise=0.20, seed=0):
    """Two interleaving half-circles."""
    torch.manual_seed(seed)
    half = n // 2
    t1, t2 = pi * torch.rand(half), pi * torch.rand(n - half)
    top = torch.stack([torch.cos(t1), torch.sin(t1)], dim=1)
    bottom = torch.stack([1 - torch.cos(t2), 0.5 - torch.sin(t2)], dim=1)
    X = torch.cat([top, bottom])
    y = torch.cat([torch.zeros(half), torch.ones(n - half)])
    return X + noise * torch.randn(n, 2), y


def make_spiral(n_per_class=200, n_classes=2, turns=1.5, noise=0.10, seed=0):
    """Interleaved spiral arms, one per class."""
    torch.manual_seed(seed)
    Xs, ys = [], []
    for c in range(n_classes):
        t = torch.linspace(0.25, 1.0, n_per_class)
        angle = 2 * pi * turns * t + 2 * pi * c / n_classes
        arm = torch.stack([t * torch.cos(angle), t * torch.sin(angle)], dim=1)
        Xs.append(arm + noise * t[:, None] * torch.randn(n_per_class, 2))
        ys.append(torch.full((n_per_class,), float(c)))
    return torch.cat(Xs), torch.cat(ys)


def wave(x):
    """The 1-D regression target used in Q3."""
    return torch.sin(2 * pi * x) + 0.5 * x


# ---------- plotting ----------
CLASS_COLORS = ["#3b75af", "#c44536"]          # class 0 blue, class 1 red


def plot_points(X, y, ax, s=18):
    for c in (0, 1):
        m = y == c
        ax.scatter(X[m, 0], X[m, 1], s=s, color=CLASS_COLORS[c],
                   edgecolor="k", linewidth=0.3, label=f"class {c}", zorder=3)


def make_grid(X, steps=200, pad=0.4):
    """A regular grid covering the data. Returns xx, yy (steps, steps) and points (steps*steps, 2)."""
    lo, hi = X.min(dim=0).values - pad, X.max(dim=0).values + pad
    xx, yy = torch.meshgrid(torch.linspace(lo[0].item(), hi[0].item(), steps),
                            torch.linspace(lo[1].item(), hi[1].item(), steps), indexing="ij")
    return xx, yy, torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)


def plot_probability_surface(prob_fn, X, y, ax, title=""):
    """Colour the plane by P(y = 1 | x) and draw the 0.5 contour."""
    xx, yy, points = make_grid(X)
    with torch.no_grad():
        p = prob_fn(points).reshape(xx.shape)
    ax.contourf(xx, yy, p, levels=21, cmap="coolwarm", vmin=0, vmax=1, alpha=0.75)
    ax.contour(xx, yy, p, levels=[0.5], colors="k", linewidths=2)
    plot_points(X, y, ax, s=14)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(title)


# ---------- training ----------
def train(model, X, y, loss_fn, epochs=2000, lr=0.01):
    """Full-batch Adam: ONE parameter update per epoch. Returns the loss of every epoch."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for _ in range(epochs):
        loss = loss_fn(model(X).squeeze(1), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return history


def accuracy(model, X, y):
    """Binary accuracy from a single-logit output: class 1 where the logit is positive."""
    with torch.no_grad():
        return ((model(X).squeeze(1) > 0).float() == y).float().mean().item()


def n_params(model):
    return sum(p.numel() for p in model.parameters())
# =================================================================================
print("helpers loaded")

---
## Q1. Why Does a Network Need a Nonlinearity? (4 marks)

Dataset: **XOR** - class 1 in quadrants I and III, so no straight line separates the classes.

A one-hidden-layer network computes

$$z_1 = XW_1^{\top} + \mathbf{b}_1, \qquad h = \mathrm{ReLU}(z_1), \qquad \hat{y} = hW_2^{\top} + b_2 .$$

Take the ReLU away and the two layers **collapse into one**:

$$\big(XW_1^{\top} + \mathbf{b}_1\big)W_2^{\top} + b_2 \;=\; X\,(W_2W_1)^{\top} + \big(W_2\mathbf{b}_1 + b_2\big),$$

which is a single `nn.Linear(D, 1)`, i.e. logistic regression again. So depth on its own buys **nothing**.
Verify that algebra numerically, then train both networks and look at what each one can draw.

**Marking split:** `collapse_two_linears` + both fits (2) + plot (1) + observation (1).


In [ ]:
def collapse_two_linears(layer1, layer2):
    """
    Two stacked nn.Linear layers with NO activation in between are a single linear map.
    Return (W, b) such that     layer2(layer1(X))  ==  X @ W.T + b     for every X.

    layer1.weight : (H, D)      layer1.bias : (H,)
    layer2.weight : (1, H)      layer2.bias : (1,)

    Returns
    -------
    W : torch.Tensor, shape (1, D)
    b : torch.Tensor, shape (1,)
    """
    # TODO: combine the two weights and the two biases using the formula in the question.
    # Matrix product in PyTorch is the @ operator.
    W = None
    b = None
    return W, b


# --- Data setup (do not modify) ---
X_xor, y_xor = make_xor(400, noise=0.25, seed=0)

# --- Numerical check of the collapse (do not modify) ---
torch.manual_seed(0)
l1, l2 = nn.Linear(2, 16), nn.Linear(16, 1)
W, b = collapse_two_linears(l1, l2)
X_check = torch.randn(5, 2)
print("collapsed W:", tuple(W.shape), "| collapsed b:", tuple(b.shape))
print("layer2(layer1(X)) == X @ W.T + b :",
      torch.allclose(l2(l1(X_check)), X_check @ W.T + b, atol=1e-6))

# TODO (a): build the two networks with nn.Sequential, both with 16 hidden units:
#     net_linear : Linear(2, 16) -> Linear(16, 1)                (no activation)
#     net_relu   : Linear(2, 16) -> nn.ReLU() -> Linear(16, 1)
# Call torch.manual_seed(0) immediately before EACH one so both start from the same weights.
net_linear = None
net_relu = None

# TODO (b): train both with train(net, X_xor, y_xor, bce, epochs=2000, lr=0.01)
# and print the training accuracy of each, using accuracy(net, X_xor, y_xor).

# TODO (Plot): one figure, two panels (figsize about (11, 5)). For each network call
#     plot_probability_surface(lambda P: torch.sigmoid(net(P)).squeeze(1), X_xor, y_xor, ax,
#                              title=f"... accuracy {acc:.2f}")

### Your Observations (Q1)
- Did the `torch.allclose` check pass? State in one line what it proves about `Linear -> Linear`.
- Report both training accuracies. What shape is the decision boundary in each of the two plots?
- Both networks have exactly the same number of parameters. Why does only one of them solve XOR?


---
## Q2. Inside the Hidden Layer - What One Neuron Computes (4 marks)

Dataset: `make_circles()` - an inner disc (class 1) inside an outer ring (class 0).

Hidden unit $j$ computes

$$h_j = \mathrm{ReLU}\big(\mathbf{w}_{1j}\cdot\mathbf{x} + b_{1j}\big),$$

so it is **exactly zero** on one side of the line $\mathbf{w}_{1j}\cdot\mathbf{x} + b_{1j} = 0$ and grows
linearly on the other side: a half-plane feature. The output layer just takes a weighted sum,
$\;\text{logit} = b_2 + \sum_j w_{2j} h_j$.

With only $H = 4$ hidden units the whole network has 17 parameters, few enough to look at **every unit
individually**. Write the network as an `nn.Module` subclass this time, so the hidden activations are reachable.

**Marking split:** `MLP` class (2) + 5-panel plot (1) + observation (1).


In [ ]:
class MLP(nn.Module):
    """One hidden layer with ReLU:  in_features -> hidden -> out_features."""

    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        # TODO: create the two layers
        #   self.hidden = nn.Linear(in_features, hidden)
        #   self.output = nn.Linear(hidden, out_features)
        pass

    def forward(self, x):
        # TODO: z1 = self.hidden(x), then h = torch.relu(z1), then return self.output(h)
        return None

    def hidden_activations(self, x):
        """Return (z1, h): the pre-activations and the post-ReLU activations of the hidden layer."""
        # TODO: return both, in that order
        return None, None


# --- Data setup and training (do not modify) ---
X_circ, y_circ = make_circles(400, noise=0.08, factor=0.45, seed=0)

torch.manual_seed(0)
tiny = MLP(2, 4, 1)                       # 2 inputs -> 4 ReLU units -> 1 logit
train(tiny, X_circ, y_circ, bce, epochs=3000, lr=0.02)

w2 = tiny.output.weight.detach().squeeze(0)
print(f"MLP 2-4-1: {n_params(tiny)} parameters, training accuracy {accuracy(tiny, X_circ, y_circ):.3f}")
print("output weights w2:", [round(v, 2) for v in w2.tolist()],
      "| output bias b2:", round(tiny.output.bias.item(), 2))

# --- The grid the hidden units are evaluated on (do not modify) ---
xx, yy, points = make_grid(X_circ)
with torch.no_grad():
    z1, h = tiny.hidden_activations(points)     # each of shape (steps*steps, 4)
print("z1:", tuple(z1.shape), " h:", tuple(h.shape))

# TODO (Plot): one figure with 5 panels, figsize about (17, 3.8).
#   Panels 0..3, for hidden unit j:
#       ax.contourf(xx, yy, h[:, j].reshape(xx.shape), levels=20, cmap="Greys")
#       ax.contour(xx, yy, z1[:, j].reshape(xx.shape), levels=[0], colors="#e69f00",
#                  linestyles="--", linewidths=2)          # the line where the unit switches on
#       plot_points(X_circ, y_circ, ax, s=6)
#       title: the unit number and its output weight w2[j]
#   Panel 4: plot_probability_surface(lambda P: torch.sigmoid(tiny(P)).squeeze(1), X_circ, y_circ, ax,
#                                     title="the whole network")

### Your Observations (Q2)
- Describe one hidden unit's panel in words: where is it exactly zero, what is the dashed line, and in which direction does it grow?
- Look at the four printed output weights `w2`. They share a sign - use that, plus the bias $b_2$, to explain why the predicted probability is highest in the middle of the plot.
- The black 0.5 contour is not a circle. What shape is it, and what single change would make it rounder?


---
## Q3. How Many Neurons, How Many Layers? (4 marks)

**Part A - width.** A one-hidden-layer ReLU network is a **piecewise-linear** function. Each unit contributes
one hinge

$$c_j(x) = w_{2j}\,\mathrm{ReLU}(w_{1j}x + b_{1j}), \qquad \hat{y}(x) = b_2 + \sum_{j=1}^{H} c_j(x),$$

and $c_j$ bends exactly once, at the **kink** $x = -b_{1j}/w_{1j}$. So $H$ units buy at most $H$ bends, and only
the kinks that land **inside the data range** are of any use. Fit `wave(x)` with widths 2 to 64 and count them.

**Part B - depth.** Then fix the problem (a 2-D spiral) and sweep depth $\times$ width, recording the
**test** accuracy and the parameter count of every network.

**Marking split:** `make_mlp` + `kink_positions` + the two sweeps (2) + plots (1) + observation (1).


In [ ]:
def make_mlp(sizes, activation=nn.ReLU):
    """
    Build an MLP from a list of layer sizes.
        make_mlp([2, 16, 16, 1])  ->  Linear(2,16), ReLU, Linear(16,16), ReLU, Linear(16,1)
    There is NO activation after the last Linear: it outputs a logit (or a regression value).
    """
    layers = []
    # TODO: walk over consecutive pairs (sizes[i], sizes[i+1]); append nn.Linear(n_in, n_out)
    # for every pair, and append activation() after every layer EXCEPT the last one.
    return nn.Sequential(*layers)


def kink_positions(model):
    """
    For a network whose first layer is nn.Linear(1, H), return the H kink locations
    x = -b1_j / w1_j as a 1-D tensor of length H.
    """
    w1 = model[0].weight.detach().squeeze(1)      # (H,)
    b1 = model[0].bias.detach()                   # (H,)
    # TODO: return the kink positions
    return None


# --- Part A data (do not modify) ---
torch.manual_seed(0)
N = 100
x_train = 2 * torch.rand(N) - 1
y_train = wave(x_train) + 0.2 * torch.randn(N)
x_test = torch.linspace(-1, 1, 400)
y_test = wave(x_test)                              # the clean function
X_train, X_test = x_train[:, None], x_test[:, None]

widths = [2, 4, 8, 32, 64]
fits, kinks_in_range, test_mse = {}, {}, {}

# TODO (a): for each width in widths:
#     torch.manual_seed(0)
#     model = make_mlp([1, width, 1])
#     train(model, X_train, y_train, mse, epochs=4000, lr=0.02)
#     with torch.no_grad(): prediction = model(X_test).squeeze(1)
#     store fits[width] = prediction
#     store test_mse[width] = ((prediction - y_test) ** 2).mean().item()
#     k = kink_positions(model); keep only those with -1 < k < 1; store kinks_in_range[width] = k
#     print the width, the number of in-range kinks and the test MSE

# TODO (Plot A): 1 x 5 panels sharing the y axis, figsize about (17, 3.6). In panel i:
#     scatter the training data, plot the true function y_test as a dashed black line,
#     plot fits[width], and mark the in-range kinks along the bottom, e.g.
#     ax.plot(k, torch.full_like(k, -1.8), "^", ms=8)
#     title: f"H = {width}: {len(k)} kinks inside, test MSE {test_mse[width]:.3f}"

In [ ]:
# --- Part B data and the heatmap helper (given, do not modify) ---
X_sp_train, y_sp_train = make_spiral(150, 2, turns=1.5, noise=0.10, seed=0)
X_sp_test,  y_sp_test  = make_spiral(150, 2, turns=1.5, noise=0.10, seed=1)
print("spiral train:", tuple(X_sp_train.shape), " test:", tuple(X_sp_test.shape))


def plot_depth_width(acc_table, size_table, depths, widths):
    fig, ax = plt.subplots(figsize=(8.5, 3.4))
    image = ax.imshow(acc_table.numpy(), cmap="viridis", vmin=0.5, vmax=1.0)
    for i in range(len(depths)):
        for j in range(len(widths)):
            ax.text(j, i, f"{acc_table[i, j]:.3f}\n{size_table[i, j]} params", ha="center", va="center",
                    color="k" if acc_table[i, j] > 0.85 else "w", fontsize=9)
    ax.set_xticks(range(len(widths)), [f"width {w}" for w in widths])
    ax.set_yticks(range(len(depths)), [f"{d} hidden layer{'s' if d > 1 else ''}" for d in depths])
    ax.grid(False)
    ax.set_title("spiral: test accuracy (and parameter count)")
    fig.colorbar(image, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.show()


depths, widths_b = [1, 2, 3], [4, 8, 16, 32]
acc_table = torch.zeros(len(depths), len(widths_b))
size_table = torch.zeros(len(depths), len(widths_b), dtype=torch.long)

# TODO (b): fill both tables. For each depth i and width j:
#     torch.manual_seed(0)
#     model = make_mlp([2] + [width] * depth + [1])
#     train(model, X_sp_train, y_sp_train, bce, epochs=1500, lr=0.01)
#     acc_table[i, j]  = accuracy(model, X_sp_test, y_sp_test)
#     size_table[i, j] = n_params(model)
# Print one line per depth as you go, so you can watch it run.

# TODO (Plot B): call plot_depth_width(acc_table, size_table, depths, widths_b)

### Your Observations (Q3)
- Part A: at which width does the fit stop being a straight line? Use the printed number of in-range kinks to explain the $H = 2$ and $H = 4$ panels.
- Part A: does the test MSE keep improving as width grows, or does it flatten? Quote two numbers.
- Part B: find one deep-and-narrow network and one shallow-and-wide network with a **similar parameter count**. Which one is more accurate?
- Part B: is 3 hidden layers always better than 2 in your heatmap? What does that say about how depth should be chosen?


---
## Q4. Activation Functions and Where Gradients Die (4 marks)

Backpropagation multiplies **one activation derivative per layer** on its way back to the input. If those
derivatives are below 1 the product shrinks geometrically. The sigmoid's derivative
$\sigma'(z) = \sigma(z)\big(1-\sigma(z)\big)$ never exceeds **0.25**, so fourteen stacked sigmoid layers scale the gradient
by at most $0.25^{14} \approx 4\times 10^{-9}$: the layers nearest the input stop learning. This is the **vanishing gradient**.

ReLU's derivative is exactly 1 for $z > 0$, so nothing shrinks. The price is that it is exactly 0 for $z < 0$:
a unit whose input is negative for *every* training point receives no gradient ever again and is **dead**.
LeakyReLU keeps a small slope on the negative side so that cannot happen.

Part A plots each activation and its derivative, taken straight from autograd. Part B **measures** the
gradient that actually reaches the first layer of a 14-hidden-layer network, one activation at a time.

**Marking split:** `activation_and_derivative` + `first_layer_grad_norm` + `dead_unit_fraction` (2) + plots (1) + observation (1).


In [ ]:
def activation_and_derivative(f, z_values):
    """
    Evaluate an elementwise activation f and its derivative f'(z) using autograd.

    The trick: for an ELEMENTWISE f, out.sum().backward() puts f'(z_i) into z.grad[i],
    because d(sum_i f(z_i)) / dz_i = f'(z_i).

    Returns
    -------
    out   : torch.Tensor - f(z), detached
    deriv : torch.Tensor - f'(z), detached
    """
    z = z_values.clone().requires_grad_(True)
    # TODO: 1) out = f(z)
    #       2) out.sum().backward()
    #       3) return out.detach(), z.grad
    return None, None


# --- Given (do not modify) ---
activations = {
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "ReLU": torch.relu,
    "LeakyReLU(0.1)": lambda z: F.leaky_relu(z, negative_slope=0.1),
}
z_grid = torch.linspace(-5, 5, 501)

# TODO (Plot A): a 2 x 4 figure sharing the x axis, figsize about (15, 5.5).
#   Top row    : f(z) against z, one column per activation, with the activation's name as the title.
#   Bottom row : f'(z) against z.
#   Give every derivative panel the SAME y limits, ax.set_ylim(0, 1.05) - with independent scales
#   sigmoid's peak of 0.25 is drawn as tall as tanh's 1.0 and the comparison is lost.
#   On the sigmoid derivative panel add ax.axhline(0.25, ls="--", color="k") and say so in the legend.

In [ ]:
# --- Given (do not modify): a deep network builder ---
def deep_net(activation_module, depth=14, width=16, in_features=2, seed=0):
    """Linear -> activation, repeated `depth` times, then a final Linear down to 1 logit."""
    torch.manual_seed(seed)
    layers, n_in = [], in_features
    for _ in range(depth):
        layers += [nn.Linear(n_in, width), activation_module()]
        n_in = width
    layers.append(nn.Linear(n_in, 1))
    return nn.Sequential(*layers)


def first_layer_grad_norm(model, X, y, loss_fn):
    """
    One forward + backward pass on the freshly initialised model. Return the L2 norm of the
    gradient that reaches the FIRST layer's weight, i.e. model[0].weight.grad.
    """
    model.zero_grad()
    # TODO: compute loss = loss_fn(model(X).squeeze(1), y), call loss.backward(),
    # and return float(model[0].weight.grad.norm())
    return None


def dead_unit_fraction(model, X):
    """
    Fraction of hidden units that output exactly 0 for EVERY row of X (only meaningful for ReLU).
    Walk the Sequential; after each activation layer, `out` holds that layer's activations (N, width).
    """
    dead = total = 0
    out = X
    with torch.no_grad():
        for layer in model:
            out = layer(out)
            if not isinstance(layer, nn.Linear):
                # TODO: add out.shape[1] to `total`, and add the number of columns of `out`
                # that are exactly 0 for every row to `dead`.
                # Hint: (out == 0).all(dim=0) is a boolean mask over the columns.
                pass
    return dead / total


# --- Data (do not modify) ---
X_m, y_m = make_moons(400, noise=0.20, seed=0)
modules = {"sigmoid": nn.Sigmoid, "tanh": nn.Tanh,
           "ReLU": nn.ReLU, "LeakyReLU(0.1)": lambda: nn.LeakyReLU(0.1)}

grad_norms, accs, dead_fracs, curves = {}, {}, {}, {}

# TODO (b): for each name, module in modules.items():
#     net = deep_net(module)                                   # 14 hidden layers of 16 units
#     grad_norms[name] = first_layer_grad_norm(net, X_m, y_m, bce)     # BEFORE any training
#     curves[name] = train(net, X_m, y_m, bce, epochs=1500, lr=0.01)
#     accs[name] = accuracy(net, X_m, y_m)
#     dead_fracs[name] = dead_unit_fraction(net, X_m)
#     print one line with the gradient norm (format {:.2e}), the FINAL training loss,
#     the accuracy and the dead-unit fraction

# TODO (Plot B): a figure with two panels, figsize about (13, 4.4).
#   Left : bar chart of grad_norms with ax.set_yscale("log") - one bar per activation.
#   Right: the four training-loss curves on the same axes, with a log y axis and a legend.

### Your Observations (Q4)
- From the derivative row: for which activations is the derivative near zero over a wide range of $z$? What does the dashed 0.25 line mean for a **deep** sigmoid network?
- Report the four first-layer gradient norms. How many orders of magnitude separate sigmoid from ReLU, and does the accuracy after training follow the same ordering?
- Report the dead-unit fraction for ReLU and for LeakyReLU. Explain the difference using the two derivative plots.
- One of the four networks never gets its training loss below 0.6931 and sits at exactly chance accuracy. What is 0.6931 in closed form, and what is such a network predicting for every input?


---
## Q5. Mini-Batches - an Epoch Is Not an Update (4 marks)

Every question so far used **full-batch** training: one gradient computed over all $N$ points, so **one**
parameter update per epoch. Real training splits the data into **mini-batches** of size $B$ and updates once
per batch, so a single pass over the data performs $\lceil N/B \rceil$ updates. Each update is noisier, because
it only sees $B$ points, but there are far more of them.

Same network, same initial weights, same number of epochs. The only thing that changes is the batch size.

**Marking split:** `train_minibatch` (2) + plots (1) + observation (1).


In [ ]:
import time

from torch.utils.data import TensorDataset, DataLoader


def train_minibatch(model, X, y, loss_fn, batch_size, epochs=100, lr=0.01):
    """
    Mini-batch Adam: one parameter update per BATCH.

    Returns
    -------
    epoch_losses : list[float] - average training loss per epoch (weighted by batch size)
    n_updates    : int         - how many times optimizer.step() was called in total
    """
    # TODO: wrap (X, y) in a TensorDataset and build a DataLoader with this batch_size
    # and shuffle=True.
    loader = None

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    epoch_losses, n_updates = [], 0

    for _ in range(epochs):
        total = 0.0
        for xb, yb in loader:
            # TODO: the usual four lines - loss on this batch, zero_grad, backward, step.
            # Then accumulate:  total += loss.item() * len(xb)  and  n_updates += 1
            pass
        epoch_losses.append(total / len(X))

    return epoch_losses, n_updates


# --- Data and the network builder (do not modify) ---
X_sp, y_sp = make_spiral(200, 2, turns=1.5, noise=0.10, seed=0)
EPOCHS = 100


def build():
    """A fresh 2-16-16-1 ReLU network, always from the same initial weights."""
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(2, 16), nn.ReLU(),
                         nn.Linear(16, 16), nn.ReLU(),
                         nn.Linear(16, 1))


model_mb, model_fb = build(), build()

# TODO (a): train model_mb with train_minibatch(..., batch_size=32, epochs=EPOCHS, lr=0.01)
# keeping both returned values, and train model_fb with train(..., epochs=EPOCHS, lr=0.01)
# (which performs exactly 1 update per epoch). Print, for each: number of updates,
# final loss and accuracy(model, X_sp, y_sp).
mb_losses, n_updates = None, None
fb_losses = None

# TODO (b): sweep batch_size over [8, 32, 128, len(X_sp)] for 50 epochs each, using a FRESH
# build() every time. Print one line per batch size: batch size, number of updates, final
# accuracy, and the wall-clock time the run took (time.perf_counter() before and after).

# TODO (Plot): two panels, figsize about (13, 4.6).
#   Left : mb_losses and fb_losses against epoch on the same axes; put the number of updates
#          in each label; a log y axis reads nicely here.
#   Right: plot_probability_surface for model_mb.

### Your Observations (Q5)
- How many updates did each run perform in the same 100 epochs, and how do the two final accuracies compare?
- The mini-batch loss curve is visibly less smooth than the full-batch one. Why?
- From the batch-size sweep: which batch size reached the best accuracy in 50 epochs? What is the cost of making the batch very small?
- In one line: why is full-batch training simply not an option on a dataset such as ImageNet?


---
### Submission Checklist
- [ ] All `# TODO` blocks completed, no `None` / `pass` placeholders remain
- [ ] All 5 figures render without errors
- [ ] All 5 Observation cells filled in
- [ ] Ran **Restart and run all** with no errors
- [ ] File renamed to `<name>_EM619_A3.ipynb`
